# Loop1, stage 01: draw the gating strategy

Three gates are drawn by hand, one after another, each on the cells that the
previous gate kept.

1. Cells, on forward scatter area against side scatter area.  This separates
   cells from debris.
2. Single cells, on forward scatter area against forward scatter height.  Two
   cells passing the laser together give a wide pulse, so they fall off the
   diagonal.
3. Live cells, on forward scatter area against the 7-AAD channel.  7-AAD only
   enters a cell whose membrane is broken, so a dead cell is bright.

The gates are stored as polygons in `gates.npz` and applied to the full dataset
in stage 02.  Drawing them here on the downsampled file keeps the plots fast.

## Inputs

| file | produced by | holds |
|---|---|---|
| `Loop1_500k.h5ad` | stage 00 | about 500,000 cells, enough to see the populations |

## Outputs

| file | holds |
|---|---|
| `gates.npz` | the three polygons and the two axis names of each |

## This notebook needs a person

It opens an interactive figure and waits for you to drag the polygon vertices.
It cannot be run unattended, and `Restart Kernel and Run All Cells` will store
whatever polygon happens to be in place.  Every other notebook in this loop is
non-interactive.

Run it only when you want to redraw the gates.  `gates.npz` is committed, so
stage 02 reproduces the published gating without this notebook.

In [ ]:
%matplotlib widget

import matplotlib.pyplot as plt
import scanpy as sc
from matplotlib import colormaps

from labcompass import DrawGate, init_gate_file

IN_SUBSET = "Loop1_500k.h5ad"
GATE_FILE = "gates.npz"

FSC_A = "FSC-A :: FSC - Area"
FSC_H = "FSC-H :: FSC - Height"
SSC_A = "SSC-A :: SSC - Area"
LIVE_DEAD = "7-AAD"

In [ ]:
adata = sc.read(IN_SUBSET)
adata

## Start a new gate file

`init_gate_file` discards any gate file already present.  `DrawGate.save_gate`
appends, so without this the new gates would be added to the old ones.

In [ ]:
init_gate_file(GATE_FILE)

## Look at the scatter plot first

The colour scale is the density of events.  The lowest bin is painted the
darkest blue of the `jet` map so that empty regions and very sparse regions look
different.

In [ ]:
import pytometry as pm

cmap = colormaps["jet"].copy()
cmap.set_bad("#00007F")

pm.pl.scatter_density(
    adata, x=FSC_A, y=SSC_A, cmap=cmap,
    x_lim=[0, 1_500_000], y_lim=[0, 1_500_000], vmin=0.001,
)

## Gate 1, cells

Enclose the main cloud and leave the low-scatter debris outside.

In [ ]:
gate = DrawGate(
    adata, x=FSC_A, y=SSC_A,
    n_vertices=8, xlim=(-1, 1_500_000), ylim=(-1, 1_500_000), high_res=True,
)

Drag the vertices, then run the cell below to close the figure, keep the cells inside the polygon, and append the polygon to the gate file.

In [ ]:
plt.close(gate.fig)
adata = gate.filter_cells(mode="any")
gate.save_gate(GATE_FILE)
print(f"gate 1 keeps {adata.n_obs:,} cells")

## Gate 2, single cells

Enclose the diagonal band.  Events below it are two cells read as one.

In [ ]:
gate = DrawGate(
    adata, x=FSC_A, y=FSC_H,
    n_vertices=8, xlim=(-1, 1_500_000), ylim=(-1, 600_000), high_res=True,
)

Drag the vertices, then run the cell below to close the figure, keep the cells inside the polygon, and append the polygon to the gate file.

In [ ]:
plt.close(gate.fig)
adata = gate.filter_cells(mode="any")
gate.save_gate(GATE_FILE)
print(f"gate 2 keeps {adata.n_obs:,} cells")

## Gate 3, live cells

Enclose the 7-AAD dim events.  Bright events are dead cells.

In [ ]:
gate = DrawGate(
    adata, x=FSC_A, y=LIVE_DEAD,
    n_vertices=4, xlim=(0, 1_500_000), ylim=(-30_000, 10_000), high_res=True,
)

Drag the vertices, then run the cell below to close the figure, keep the cells inside the polygon, and append the polygon to the gate file.

In [ ]:
plt.close(gate.fig)
adata = gate.filter_cells(mode="any")
gate.save_gate(GATE_FILE)
print(f"gate 3 keeps {adata.n_obs:,} cells")

## Check what was saved

In [ ]:
from labcompass import read_gate_file

polygons, x_cols, y_cols = read_gate_file(GATE_FILE)
for i, (poly, x, y) in enumerate(zip(polygons, x_cols, y_cols), start=1):
    print(f"gate {i}: {len(poly)} vertices on {x} against {y}")